# 03 — Storage anomaly hunt

Looks for evidence consistent with the **Defender for Storage** scenarios
(S-STO-02 / 03 / 04 / 05 / 08): anonymous access, Tor / TI IPs, mass
extraction, mass deletion. Requires `StorageBlobLogs` diagnostic setting on
the storage account → LAW.


In [ ]:
"""Shared setup — run me first.
Connects to Log Analytics via DefaultAzureCredential (uses az login, env vars, or managed identity).
"""
import os
import pandas as pd
from datetime import timedelta
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

WORKSPACE_ID = os.environ.get("LAW_WORKSPACE_ID")  # GUID of the Log Analytics workspace
assert WORKSPACE_ID, "Set LAW_WORKSPACE_ID env var (workspace customerId GUID, not full ARM id)"

client = LogsQueryClient(DefaultAzureCredential())

def kql(query: str, hours: int = 24) -> pd.DataFrame:
    """Run a KQL query against the workspace and return the first table as a DataFrame."""
    resp = client.query_workspace(WORKSPACE_ID, query, timespan=timedelta(hours=hours))
    if resp.status == LogsQueryStatus.PARTIAL:
        print("WARNING: partial result -", resp.partial_error)
        tables = resp.partial_data
    elif resp.status == LogsQueryStatus.SUCCESS:
        tables = resp.tables
    else:
        raise RuntimeError(f"Query failed: {resp}")
    if not tables:
        return pd.DataFrame()
    t = tables[0]
    return pd.DataFrame(t.rows, columns=t.columns)

pd.set_option("display.max_colwidth", 120)
print("Workspace:", WORKSPACE_ID)


## Anonymous successful access

In [ ]:
df = kql("""
StorageBlobLogs
| where TimeGenerated > ago(24h)
| where AuthenticationType == "Anonymous"
| where StatusText == "Success"
| summarize Count=count(), Ops=make_set(OperationName, 5), Files=make_set(Uri, 5)
          by AccountName, CallerIpAddress
| order by Count desc
""")
df


## Mass extraction — top callers by bytes read

In [ ]:
df = kql("""
StorageBlobLogs
| where TimeGenerated > ago(24h)
| where OperationName in ("GetBlob", "GetBlobProperties")
| summarize BytesRead=sum(ResponseBodySize), Ops=count()
          by CallerIpAddress, AccountName, RequesterObjectId
| where BytesRead > 0
| order by BytesRead desc
| take 20
""")
df


## Mass deletion

In [ ]:
df = kql("""
StorageBlobLogs
| where TimeGenerated > ago(24h)
| where OperationName in ("DeleteBlob", "DeleteContainer")
| summarize Count=count() by AccountName, CallerIpAddress, RequesterObjectId
| where Count > 10
| order by Count desc
""")
df


## Correlate caller IPs with Sentinel TI

In [ ]:
df = kql("""
let ti = ThreatIntelligenceIndicator
       | where ExpirationDateTime > now() and Active == true and isnotempty(NetworkIP)
       | distinct NetworkIP;
StorageBlobLogs
| where TimeGenerated > ago(24h)
| where CallerIpAddress in (ti)
| summarize Count=count(), Ops=make_set(OperationName, 5)
          by AccountName, CallerIpAddress
""")
df
